In [1]:
import os, re, warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT_DIR = "Code Outputs/Climate EDA Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
def out(n): return os.path.join(OUTPUT_DIR, n)

# Load climate and levels
clim = pd.read_excel("Code Outputs/Climate Data Extraction Outputs/Lake_Climate_Monthly.xlsx"); clim["Date"] = pd.to_datetime(clim["Date"])
lev  = pd.read_excel("Code Outputs/Gap Interpolation Outputs/Unified_Interpolated_Levels.xlsx"); lev["Date"] = pd.to_datetime(lev["Date"])
LAKES = sorted(clim["Reservoir"].unique())

def deseasonalise(s):
    """Subtract the calendar-month climatology"""
    s = s.copy()
    return s - s.groupby(s.index.month).transform("mean")

# wide level + deseasonalised level anomaly
level_w = lev.pivot(index="Date", columns="Reservoir", values="Level_m").sort_index().asfreq("MS")
lvl_anom = level_w.apply(deseasonalise)

# Climate series per lake

fig, axes = plt.subplots(len(LAKES), 1, figsize=(14, 2.6 * len(LAKES)), sharex=True)
for ax, lk in zip(axes, LAKES):
    g = clim[clim.Reservoir == lk].set_index("Date").sort_index()
    ax.bar(g.index, g["precip_mm"], width=20, color="tab:blue", alpha=.5, label="precip (mm/mo)")
    ax.plot(g.index, g["pet_mm"], color="tab:red", lw=.8, label="PET (mm/mo)")
    ax.set_title(lk, fontweight="bold"); ax.set_ylabel("mm/mo")
    if lk == LAKES[0]: ax.legend(fontsize=8, loc="upper right")
plt.suptitle("Monthly Precipitation and Potential Evaporation by Lake", fontweight="bold", y=.999)
plt.tight_layout(); plt.savefig(out("CLIM_01_series.png"), dpi=800); plt.close()

# Mean precip by month
rain_clim = pd.DataFrame({
    lk: clim[clim.Reservoir == lk].assign(m=lambda d: d.Date.dt.month)
            .groupby("m")["precip_mm"].mean()
    for lk in LAKES})
rain_clim.to_csv(out("CLIM_02_rain_climatology.csv"))
plt.figure(figsize=(11, 6))
for lk in LAKES:
    plt.plot(range(1, 13), rain_clim[lk], marker="o", label=lk)
plt.xticks(range(1, 13), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
plt.title("Mean Monthly Rainfall by Lake (seasonal regime)", fontweight="bold")
plt.ylabel("precip (mm/month)"); plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.savefig(out("CLIM_02_rain_climatology.png"), dpi=800); plt.close()

# lagged correlation 
def lead_corr(level_anom, driver_anom, max_lag=12):
    j = level_anom.dropna().index.intersection(driver_anom.dropna().index)
    L, D = level_anom.loc[j], driver_anom.loc[j]
    return {k: L.corr(D.shift(k)) for k in range(0, max_lag + 1)}

rows = []
fig, axes = plt.subplots(4, 2, figsize=(15, 16)); axes = axes.flatten()
for i, lk in enumerate(LAKES):
    g = clim[clim.Reservoir == lk].set_index("Date").sort_index()
    precip_anom = deseasonalise(g["precip_mm"].asfreq("MS"))
    wb_anom     = deseasonalise(g["water_balance_mm"].asfreq("MS"))
    lc_p  = lead_corr(lvl_anom[lk], precip_anom)
    lc_wb = lead_corr(lvl_anom[lk], wb_anom)
    bestk = max(lc_p, key=lambda k: abs(lc_p[k]))
    rows.append({"Lake": lk, "precip_peak_lag_m": bestk,
                 "precip_peak_corr": round(lc_p[bestk], 3),
                 "wb_peak_lag_m": max(lc_wb, key=lambda k: abs(lc_wb[k])),
                 "wb_peak_corr": round(max(lc_wb.values(), key=abs), 3)})
    ax = axes[i]
    ax.plot(list(lc_p), list(lc_p.values()), marker="o", label="precip")
    ax.plot(list(lc_wb), list(lc_wb.values()), marker="s", label="water balance")
    ax.axhline(0, color="k", lw=.8); ax.set_title(lk, fontweight="bold")
    ax.set_xlabel("driver leads level by k months"); ax.set_ylabel("corr")
    if i == 0: ax.legend(fontsize=8)
axes[7].set_visible(False)
plt.suptitle("Lagged Correlation: climate driver leading lake-level anomaly", fontweight="bold", y=.995)
plt.tight_layout(); plt.savefig(out("CLIM_03_lagged_corr.png"), dpi=800); plt.close()
pd.DataFrame(rows).to_csv(out("CLIM_03_lead_lag_summary.csv"), index=False)
print("=== 3. PRECIP / WATER-BALANCE LEAD-LAG vs LEVEL ===")
print(pd.DataFrame(rows).to_string(index=False))

# water balance vs level
fig, axes = plt.subplots(4, 2, figsize=(15, 16)); axes = axes.flatten()
for i, lk in enumerate(LAKES):
    g = clim[clim.Reservoir == lk].set_index("Date").sort_index()
    wb = deseasonalise(g["water_balance_mm"].asfreq("MS"))
    z = lambda s: (s - s.mean()) / s.std()
    ax = axes[i]
    ax.plot(lvl_anom[lk].index, z(lvl_anom[lk]), label="level anomaly", lw=1)
    ax.plot(wb.index, z(wb), label="water-balance anomaly", lw=1, alpha=.7)
    ax.axhline(0, color="k", lw=.6); ax.set_title(lk, fontweight="bold")
    if i == 0: ax.legend(fontsize=8)
axes[7].set_visible(False)
plt.suptitle("Standardised Water-Balance Anomaly vs Level Anomaly", fontweight="bold", y=.995)
plt.tight_layout(); plt.savefig(out("CLIM_04_waterbalance_vs_level.png"), dpi=800); plt.close()

# Climate indices

def parse_index(path):
    if not os.path.exists(path):
        print(f"  [missing] {path} - skipping index"); return None
    recs = []
    for line in open(path):
        toks = re.split(r"[,\s]+", line.strip())
        if not toks or toks == [""]:
            continue
        # Format A: first token is a YYYY-MM-DD (or YYYY/MM/DD) date
        m = re.match(r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})$", toks[0])
        if m and len(toks) >= 2:
            try: v = float(toks[1])
            except ValueError: continue
            recs.append((pd.Timestamp(int(m[1]), int(m[2]), 1), v)); continue
        # Format B: 13 tokens, first is a 4-digit year, rest numeric
        if len(toks) == 13 and re.fullmatch(r"\d{4}", toks[0]):
            try: vals = [float(t) for t in toks[1:]]
            except ValueError: continue
            for mo, v in enumerate(vals, 1):
                recs.append((pd.Timestamp(int(toks[0]), mo, 1), v))
    if not recs:
        print(f"  [warn] no rows parsed from {path}"); return None
    s = pd.Series(dict(recs)).sort_index()
    s[s < -90] = np.nan                       # handles -9999 and -99.99
    return s.asfreq("MS")

dmi  = parse_index("Climate Indices/dmi.csv")
nino = parse_index("Climate Indices/nino34.csv")
if nino is not None:
    nino = deseasonalise(nino)

idx_rows = []
fig, ax = plt.subplots(figsize=(15, 5))
for name, series, c in [("DMI (IOD)", dmi, "tab:green"), ("Nino3.4 anom (ENSO)", nino, "tab:red")]:
        if series is not None:
            s = series.loc["1992":]                     # crop to the study period
            ax.plot(s.index, s, label=name, color=c, lw=1)
ax.axhline(0, color="k", lw=.8); ax.legend(); ax.set_title("Climate Indices", fontweight="bold")
plt.tight_layout(); plt.savefig(out("CLIM_05_indices.png"), dpi=800); plt.close()

for lk in LAKES:
    r = {"Lake": lk}
    for name, series in [("DMI", dmi), ("Nino34", nino)]:
        if series is not None:
            lc = lead_corr(lvl_anom[lk], series)
            bk = max(lc, key=lambda k: abs(lc[k]))
            r[f"{name}_peak_lag_m"] = bk; r[f"{name}_peak_corr"] = round(lc[bk], 3)
    idx_rows.append(r)
pd.DataFrame(idx_rows).to_csv(out("CLIM_05_index_leadlag.csv"), index=False)
print("\n=== 5. CLIMATE INDEX LEAD-LAG vs LEVEL ANOMALY ===")
print(pd.DataFrame(idx_rows).to_string(index=False))


print("\nClimate EDA complete. Tables: CLIM_*.csv  |  Figures: CLIM_*.png")

=== 3. PRECIP / WATER-BALANCE LEAD-LAG vs LEVEL ===
           Lake  precip_peak_lag_m  precip_peak_corr  wb_peak_lag_m  wb_peak_corr
    Lake Albert                  0            -0.075              0        -0.098
    Lake Edward                  8             0.267              8         0.268
      Lake Kivu                 10             0.199             10         0.203
    Lake Malawi                  3             0.150              3         0.143
Lake Tanganyika                 12             0.237             12         0.225
   Lake Turkana                  9             0.083              9         0.058
  Lake Victoria                  9             0.113              9         0.114

=== 5. CLIMATE INDEX LEAD-LAG vs LEVEL ANOMALY ===
           Lake  DMI_peak_lag_m  DMI_peak_corr  Nino34_peak_lag_m  Nino34_peak_corr
    Lake Albert              12          0.320                 11             0.154
    Lake Edward              12          0.279                  6       